In [5]:
#!/usr/bin/env python3
# --- PATCHCORE-VIT + REAL-STAMP VIEWER PIPELINE ---

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import nn
import timm

from seticore import viewer

# suppress anoying matplotlib warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="seticore.viewer")

# ----------------- CONFIG -----------------
RESAMPLED_GLOB = "/datax/scratch/jaym/resampled_mk_stamps/*.npy"
MANIFEST_CSV   = "/datax/scratch/jaym/resampled_mk_stamps/manifest.csv"
SHOW_TOP_K     = 12                  # how many anomalies to always show
PATCH          = 16
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
# -----------------------------------------

# ----------------- LOAD RESAMPLED -----------------
paths = sorted(glob.glob(RESAMPLED_GLOB))
stamps = []
for p in paths:
    try:
        stamps.append(np.load(p))
    except Exception as e:
        print(f"[WARN] Error loading {p}: {e}")
print(f"Loaded {len(stamps)} resampled stamps.")

# ----------------- HELPERS -----------------
def get_full_stamp(arr):
    # arr: (T, F, P, A)
    return arr.sum(axis=2) if arr.shape[2] > 1 else arr[:, :, 0, :]

def pad_or_crop_stamp(a, target_shape=(16, 640, 64)):
    T, F, A = a.shape
    t_pad = max(0, target_shape[0] - T)
    f_pad = max(0, target_shape[1] - F)
    a_pad = max(0, target_shape[2] - A)
    pad_width = ((0, t_pad), (0, f_pad), (0, a_pad))
    padded = np.pad(a, pad_width, mode='constant')
    return padded[:target_shape[0], :target_shape[1], :target_shape[2]]

def make_big_image_vertical(stamp_tfpa):
    x = stamp_tfpa.astype("float32")
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + 1e-6
    x = (x - med) / (1.4826 * mad)
    # vertically stack antennas: (T*A, F)
    return np.concatenate([x[:, :, i] for i in range(x.shape[2])], axis=0)

def split_vertical_antennas(img2d, n_ant=64):
    total_H, W = img2d.shape
    T = total_H // n_ant
    tiles = [img2d[i*T:(i+1)*T, :] for i in range(n_ant)]
    return tiles, T, W

def show_anomap_grid_from_vertical(amap2d, nrow=8, ncol=8, title=""):
    tiles, T, W = split_vertical_antennas(amap2d, n_ant=nrow*ncol)
    vmin, vmax = np.percentile(amap2d, 2), np.percentile(amap2d, 98)
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*1.2, nrow*1.2))
    for a, tile in enumerate(tiles):
        ax = axes[a//ncol, a % ncol]
        ax.imshow(tile, aspect='auto', origin='lower', cmap='inferno', vmin=vmin, vmax=vmax)
        ax.set_title(f"Ant {a}", fontsize=6)
        ax.axis('off')
    if title:
        plt.suptitle(title)
    plt.tight_layout(); plt.show()

# Build vertically-stacked images for ViT
big_images = [make_big_image_vertical(pad_or_crop_stamp(get_full_stamp(s))) for s in stamps]

# ----------------- ViT FEATURE EXTRACTOR -----------------
class ViTFeat(nn.Module):
    def __init__(self, img_size, model_name='vit_small_patch16_224', patch_size=16):
        super().__init__()
        H, W = img_size
        self.patch_size = patch_size
        self.h = H // patch_size
        self.w = W // patch_size
        self.num_patches = self.h * self.w
        self.vit = timm.create_model(model_name, pretrained=True, num_classes=0,
                                     img_size=(H, W), patch_size=patch_size)
        if hasattr(self.vit, 'patch_embed'):
            self.vit.patch_embed.img_size = (H, W)
            self.vit.patch_embed.strict_img_size = False
        for p in self.vit.parameters():
            p.requires_grad = False

    def forward(self, x):
        x3 = x.repeat(1, 3, 1, 1)           # 1->3 channels
        z = self.vit.forward_features(x3)
        if z.ndim == 4:
            B, D, h, w = z.shape
            z = z.permute(0, 2, 3, 1).reshape(B, h * w, D)
        else:
            B, N, D = z.shape
            if N != self.num_patches:
                z = z[:, -self.num_patches:, :]
        if z.shape[1] > self.num_patches:
            z = z[:, :self.num_patches, :]
        return z

# ----------------- PATCH FILTERING & MEMORY -----------------
def robust_z(x):
    return (x - np.median(x)) / (1.4826 * (np.median(np.abs(x - np.median(x))) + 1e-6))

def mask_obvious(x, zthr=4.0):
    z = robust_z(x)
    mask = (np.abs(z) > zthr)
    row_bad, col_bad = (np.mean(mask, axis=1) > 0.05), (np.mean(mask, axis=0) > 0.05)
    mask[row_bad, :], mask[:, col_bad] = True, True
    return mask.astype(np.uint8)

def keep_patch_indices(mask, patch=16):
    Hc, Wc = (mask.shape[0] // patch) * patch, (mask.shape[1] // patch) * patch
    mask = mask[:Hc, :Wc]
    h, w = Hc // patch, Wc // patch
    keep = [i * w + j for i in range(h) for j in range(w)
            if mask[i*patch:(i+1)*patch, j*patch:(j+1)*patch].mean() <= 0.1]
    return keep, Hc, Wc

@torch.no_grad()
def build_pseudo_memory(vit_feat, images, patch=16, target_patches=200_000, device='cuda'):
    feats = []
    for img2d in tqdm(images, desc="Building memory"):
        mask = mask_obvious(img2d)
        keep, Hc, Wc = keep_patch_indices(mask, patch)
        if not keep:
            continue
        img_crop = img2d[:Hc, :Wc]
        x = torch.from_numpy(img_crop).unsqueeze(0).unsqueeze(0).to(device)
        z = vit_feat(x)[0]
        expected = (Hc // patch) * (Wc // patch)
        if z.shape[0] != expected:
            z = z[-expected:, :]
        feats.append(z[keep].cpu())
    all_feats = torch.cat(feats, dim=0)
    if len(all_feats) > target_patches:
        all_feats = all_feats[torch.randperm(len(all_feats))[:target_patches]]
    return all_feats

# ----------------- SCORING -----------------
@torch.no_grad()
def patch_knn_distance(Q, M, k=1, device='cuda'):
    Qn, Mn = F.normalize(Q.to(device), dim=1), F.normalize(M.to(device), dim=1)
    return 1 - torch.topk(Qn @ Mn.T, k=k, dim=1).values.mean(dim=1)

def unpatchify_scalar(dists, H, W, patch=16):
    h, w = H // patch, W // patch
    amap = dists.view(1, 1, h, w)
    return F.interpolate(amap, size=(H, W), mode='nearest')[0, 0].cpu().numpy()

@torch.no_grad()
def score_image(img2d, vit_feat, memory, patch=16, topk_frac=0.01, device='cuda'):
    Hc, Wc = (img2d.shape[0] // patch) * patch, (img2d.shape[1] // patch) * patch
    img_crop = img2d[:Hc, :Wc]
    x = torch.from_numpy(img_crop).unsqueeze(0).unsqueeze(0).to(device)
    z = vit_feat(x)[0]
    expected = (Hc // patch) * (Wc // patch)
    if z.shape[0] != expected:
        z = z[-expected:, :]
    dists = patch_knn_distance(z, memory, k=1, device=device)
    amap = unpatchify_scalar(dists, Hc, Wc, patch)
    k = max(1, int(amap.size * topk_frac))
    return amap, float(np.mean(np.partition(amap.flatten(), -k)[-k:]))

# ----------------- RUN PATCHCORE -----------------
PATCH = PATCH
Hc, Wc = (big_images[0].shape[0] // PATCH) * PATCH, (big_images[0].shape[1] // PATCH) * PATCH
vit_feat = ViTFeat(img_size=(Hc, Wc), patch_size=PATCH).to(DEVICE)

memory = build_pseudo_memory(vit_feat, big_images, patch=PATCH, device=DEVICE)
if memory.numel() == 0:
    raise RuntimeError("Memory bank is empty; relax mask or provide more images.")

torch.save(memory, "patchcore_memory.pt")

scores = []
amaps = []
for img in tqdm(big_images, desc="Scoring"):
    amap, s = score_image(img, vit_feat, memory, patch=PATCH, device=DEVICE)
    amaps.append(amap)
    scores.append(s)

scores = np.asarray(scores, dtype=np.float32)
order = np.argsort(scores)[::-1]

print("Top anomalies:")
for r, idx in enumerate(order[:SHOW_TOP_K], 1):
    print(f"  #{r:02d} idx={idx} score={scores[idx]:.4f}")

# ----------------- MAP TO REAL STAMPS -----------------
# manifest has columns: out_file (original .stamps), resampled_file (this .npy), resampled_index (inner index)
manifest = pd.read_csv(MANIFEST_CSV)

# Build a DF of top anomalies in anomaly order
top_df = pd.DataFrame({
    "resampled_file": [paths[i] for i in order],   # full path strings
    "rank": np.arange(len(order), dtype=int)
})

# Join to get original paths + indices; keep only rows that successfully map
joined = top_df.merge(manifest, on="resampled_file", how="left").dropna(subset=["out_file", "resampled_index"])
joined = joined.sort_values("rank", ascending=True).reset_index(drop=True)

# Array of original .stamps paths in anomaly order (duplicates possible if multiple anomalies are from same file)
real_paths_anomalous = joined["out_file"].to_numpy()
# Also keep (path, index) pairs — safer, because one container can hold multiple stamps
real_pairs_anomalous = list(zip(joined["out_file"].tolist(), joined["resampled_index"].astype(int).tolist()))

print(f"\nCollected {len(real_paths_anomalous)} mapped anomalies to original .stamps.")

# ----------------- DISPLAY REAL STAMPS (ALL-ANTENNAS) -----------------
def load_stamp_by_index(stamps_path: str, idx: int):
    # Fetch the idx-th stamp without materializing the whole list
    for j, st in enumerate(viewer.read_stamps(stamps_path, find_recipe=True)):
        if j == idx:
            return st
    raise IndexError(f"Index {idx} out of range for {stamps_path}")

def show_real_stamp(stamps_path: str, idx: int, title: str):
    st = load_stamp_by_index(stamps_path, idx)
    st.show_antennas(title=title)
    plt.show()

print("\nDisplaying top real-stamp anomalies...")
for k, (src_path, inner_idx) in enumerate(real_pairs_anomalous[:SHOW_TOP_K], 1):
    title = f"[#{k:02d}] {os.path.basename(src_path)}  (idx {inner_idx})  score={scores[order[k-1]]:.4f}"
    print(f"  Viewing {title}")
    try:
        #show_real_stamp(src_path, inner_idx, title)
        pass
    except Exception as e:
        print(f"[WARN] Could not display {src_path} [idx {inner_idx}]: {e}")

# ----------------- OUTPUTS YOU CAN USE LATER -----------------
# real_paths_anomalous: np.ndarray of original .stamps paths in anomaly order
# real_pairs_anomalous: list of (original .stamps path, inner index) in anomaly order

# If you want to persist them for later:
# np.save("real_paths_anomalous.npy", real_paths_anomalous)
# np.save("real_pairs_anomalous.npy", np.array(real_pairs_anomalous, dtype=object))


Loaded 323 resampled stamps.


Scoring: 100%|██████████| 323/323 [01:21<00:00,  3.94it/s]

Top anomalies:
  #01 idx=59 score=0.6998
  #02 idx=61 score=0.6982
  #03 idx=46 score=0.6960
  #04 idx=44 score=0.6923
  #05 idx=47 score=0.6917
  #06 idx=49 score=0.6846
  #07 idx=48 score=0.6687
  #08 idx=41 score=0.6632
  #09 idx=173 score=0.6521
  #10 idx=67 score=0.6430
  #11 idx=63 score=0.6397
  #12 idx=254 score=0.6390

Collected 323 mapped anomalies to original .stamps.

Displaying top real-stamp anomalies...
  Viewing [#01] stamp_0027.stamps  (idx 0)  score=0.6998
  Viewing [#02] stamp_0027.stamps  (idx 2)  score=0.6982
  Viewing [#03] stamp_0022.stamps  (idx 2)  score=0.6960
  Viewing [#04] stamp_0022.stamps  (idx 0)  score=0.6923
  Viewing [#05] stamp_0023.stamps  (idx 0)  score=0.6917
  Viewing [#06] stamp_0023.stamps  (idx 2)  score=0.6846
  Viewing [#07] stamp_0023.stamps  (idx 1)  score=0.6687
  Viewing [#08] stamp_0021.stamps  (idx 0)  score=0.6632
  Viewing [#09] stamp_0075.stamps  (idx 2)  score=0.6521
  Viewing [#10] stamp_0029.stamps  (idx 2)  score=0.6430
  Viewin

In [2]:
from collections import defaultdict
from seticore import viewer

# group indices by file
want = defaultdict(set)
for src_path, inner_idx in real_pairs_anomalous[:30]:
    want[src_path].add(int(inner_idx))

stamps = []
stamp_file_map = []

for src_path, idx_set in want.items():
    for j, st in enumerate(viewer.read_stamps(src_path, find_recipe=True)):
        if j in idx_set:
            stamps.append(st)
            stamp_file_map.append((src_path, j))
        # optional early exit if we've collected all desired indices for this file
        if len(idx_set) == sum(1 for _, jj in stamp_file_map if _ == src_path and jj in idx_set):
            break

In [4]:
for i in range(len(stamps[:30])):
    print(f'Anomaly rank #{i + 1}. RA = {stamps[i].stamp.ra}, DEC = {stamps[i].stamp.dec}, TSTART = {stamps[i].stamp.tstart}, Drift Rate = {stamps[i].stamp.signal.driftRate}, Frequency = {stamps[i].stamp.signal.frequency}, SNR = {stamps[i].stamp.signal.snr}')

Anomaly rank #1. RA = 0.711961111111111, DEC = 12.782666666666668, TSTART = 1727304956.8597045, Drift Rate = -0.0040743422744974, Frequency = 955.3042094707489, SNR = 174.28176879882812
Anomaly rank #2. RA = 0.711961111111111, DEC = 12.782666666666668, TSTART = 1727304956.8597045, Drift Rate = -0.0020371711372487, Frequency = 953.0984146595001, SNR = 74.16809844970703
Anomaly rank #3. RA = 0.711961111111111, DEC = 12.782666666666668, TSTART = 1727304956.8597045, Drift Rate = -0.0040743422744974, Frequency = 953.7717131972313, SNR = 151.7667236328125
Anomaly rank #4. RA = 0.711961111111111, DEC = 12.782666666666668, TSTART = 1727306294.8426805, Drift Rate = -0.0020371711372487, Frequency = 940.4586006402969, SNR = 490.1156921386719
Anomaly rank #5. RA = 0.711961111111111, DEC = 12.782666666666668, TSTART = 1727306294.8426805, Drift Rate = -0.0040743422744974, Frequency = 939.0505340099335, SNR = 490.0439453125
Anomaly rank #6. RA = 0.711961111111111, DEC = 12.782666666666668, TSTART = 1